# Drone Evaluation Metrics Notebook

This notebook analyzes richer evaluation metrics for simulation and training outputs:
- success rate vs initial-condition distribution
- time-to-capture distribution
- phase portrait / trajectory views
- regime map on a grid of initial conditions


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from analysis import (
    plot_phase_portrait_samples,
    plot_regime_map,
    plot_success_vs_initial_distance,
    plot_time_to_capture_by_policy,
    plot_trajectory_grid,
    run_initial_condition_grid,
    summarize_policy_eval_figure,
)
from env import BlueEvasivePolicy, DroneEnv, RedPursuitPolicy, load_env_config

print('Root:', ROOT)

In [ ]:
# Configuration knobs
MODE = 'CONTINUOUS'
OUTPUT_DIR = ROOT / 'drone_data'
GRID_N = 15

train_cfg = load_env_config(profile='train', overrides={'OUTPUT_DIR': str(OUTPUT_DIR)})
sim_cfg = load_env_config(profile='simulate', overrides={'OUTPUT_DIR': str(OUTPUT_DIR)})

sim_step_path = OUTPUT_DIR / 'drone_dataset.csv'
sim_ep_path = OUTPUT_DIR / 'drone_dataset_episodes.csv'
train_step_path = OUTPUT_DIR / f'evaluation_results_{MODE}.csv'
train_ep_path = OUTPUT_DIR / f'evaluation_episode_summary_{MODE}.csv'

sim_step_df = pd.read_csv(sim_step_path) if sim_step_path.exists() else pd.DataFrame()
sim_ep_df = pd.read_csv(sim_ep_path) if sim_ep_path.exists() else pd.DataFrame()
train_step_df = pd.read_csv(train_step_path) if train_step_path.exists() else pd.DataFrame()
train_ep_df = pd.read_csv(train_ep_path) if train_ep_path.exists() else pd.DataFrame()

print('simulate steps:', len(sim_step_df), 'simulate episodes:', len(sim_ep_df))
print('train eval steps:', len(train_step_df), 'train eval episodes:', len(train_ep_df))

In [ ]:
# Combine episode-level views across frameworks
episode_frames = []
if not sim_ep_df.empty:
    episode_frames.append(sim_ep_df.copy())
if not train_ep_df.empty:
    episode_frames.append(train_ep_df.copy())

all_ep_df = pd.concat(episode_frames, ignore_index=True) if episode_frames else pd.DataFrame()
all_ep_df.head()

In [ ]:
# Core requested metrics: success-vs-init and time-to-capture
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_success_vs_initial_distance(all_ep_df, ax=axes[0], n_bins=10, title='Success vs Initial Distance')
plot_time_to_capture_by_policy(all_ep_df, ax=axes[1], title='Time-to-Capture by Policy')
fig.tight_layout()
plt.show()

In [ ]:
# Better overall summary dashboard
_ = summarize_policy_eval_figure(all_ep_df, title_prefix='Combined')
plt.show()

In [ ]:
# Phase portraits from step-level logs
step_frames = []
if not sim_step_df.empty:
    step_frames.append(sim_step_df.copy())
if not train_step_df.empty:
    step_frames.append(train_step_df.copy())
all_step_df = pd.concat(step_frames, ignore_index=True) if step_frames else pd.DataFrame()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
plot_phase_portrait_samples(all_step_df, policy_name='BC+RL', max_episodes=12, ax=ax, title='Phase Portrait (x vs vx)')
fig.tight_layout()
plt.show()

In [ ]:
# Regime map + trajectory grid on deterministic initial-condition sweep
try:
    import torch
    from ppo import PPOAgent
except ImportError:
    torch = None
    PPOAgent = None

env = DroneEnv(mode=MODE, config=train_cfg)
red_policy = RedPursuitPolicy(train_cfg)

# Prefer RL policy when available, fallback to heuristic expert
blue_policy = BlueEvasivePolicy(train_cfg)
rl_model_path = OUTPUT_DIR / f'rl_model_{MODE}.pth'

if PPOAgent is not None and torch is not None and rl_model_path.exists():
    state_dim = env.get_state_dim()
    action_dim = env.get_action_dim()
    rl_agent = PPOAgent(state_dim, action_dim, MODE)
    rl_agent.model.load_state_dict(torch.load(rl_model_path, map_location='cpu'))
    rl_agent.model.eval()

    class RLPolicyWrapper:
        def __init__(self, agent, scale):
            self.agent = agent
            self.scale = scale

        def get_action(self, obs, agent_type='blue'):
            state_vec = env.get_flat_state(obs)
            s = torch.tensor(state_vec, dtype=torch.float32)
            with torch.no_grad():
                action, _, _ = self.agent.model.get_action(s, deterministic=True)
            if MODE == 'DISCRETE':
                return int(action.item())
            return action.detach().cpu().numpy() * self.scale

    blue_policy = RLPolicyWrapper(rl_agent, train_cfg.BLUE_MAX_ACCEL)

regime_df, traj_map = run_initial_condition_grid(
    env,
    blue_policy=blue_policy,
    red_policy=red_policy,
    grid_n=GRID_N,
    fixed_red_pos=(train_cfg.ARENA_SIZE * 0.5, train_cfg.ARENA_SIZE * 0.5),
    include_trajectories=True,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plot_regime_map(regime_df, arena_size=train_cfg.ARENA_SIZE, ax=axes[0], title='Regime Map (Capture=1)')
plot_trajectory_grid(traj_map, arena_size=train_cfg.ARENA_SIZE, ax=axes[1], title='Trajectory Grid Samples')
fig.tight_layout()
plt.show()

regime_df.head()

In [ ]:
# Optional: persist regime map table
regime_out = OUTPUT_DIR / f'regime_map_{MODE}.csv'
if 'regime_df' in locals() and isinstance(regime_df, pd.DataFrame) and not regime_df.empty:
    regime_df.to_csv(regime_out, index=False)
    print('Saved', regime_out)
else:
    print('No regime map data to save.')